# silentchurn - Step 2 runbook (labels, series, benchmark, model, Power BI package)
Run the cells in order, one at a time. Cell 1 replaces the old Cell 1. Every step is resumable: if the session dies, re-run Cell 1 and then the step you were on.

## CELL 1 (REPLACES the old Cell 1) - setup that also fixes a flat GitHub layout

In [ ]:
# ===== CELL 1 (replacement): SETUP =====
GITHUB_USER = "RoshiniMadisetty"
REPO_NAME   = "EarlyDetectionOfCustomerAttrition"
SAMPLE_PCT  = 10

import os, subprocess, shutil, glob
from google.colab import drive
drive.mount('/content/drive')

RAW  = '/content/raw'
OUT  = '/content/drive/MyDrive/silentchurn_data'
SD   = f'{OUT}/sample{SAMPLE_PCT}'        # sampled Parquet built in step 1
WORK = f'{OUT}/work'                      # all analysis outputs (on Drive, survive session resets)
REPO = f'/content/{REPO_NAME}'
os.makedirs(RAW, exist_ok=True); os.makedirs(WORK, exist_ok=True)

if os.path.isdir(REPO):
    r = subprocess.run(['git', '-C', REPO, 'pull'], capture_output=True, text=True)
else:
    r = subprocess.run(['git', 'clone', f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git', REPO],
                       capture_output=True, text=True)
print(r.stdout, r.stderr)
assert r.returncode == 0, "git failed: check GITHUB_USER, repo name, and that the repo is public"
os.chdir(REPO)

# If the .py files were uploaded loose in the repo root, build the folders the code needs.
os.makedirs('silentchurn', exist_ok=True); os.makedirs('tests', exist_ok=True)
for f in glob.glob('*.py'):
    shutil.copy(f, f'tests/{f}' if f.startswith('test_') else f'silentchurn/{f}')
need = ['__init__', 'config', 'ingest', 'synth', 'events', 'series', 'smoothers', 'benchmark', 'model',
        'exports', 'figures', 'usability', 'pipeline']
missing = [m for m in need if not os.path.isfile(f'silentchurn/{m}.py')]
assert not missing, f"These files are missing from the repo: {missing}. Upload them to GitHub and re-run this cell."
assert os.path.isdir(SD), f"{SD} not found: step 1 output is missing (is Drive mounted?)"
print("repo OK; sample dir OK:", sorted(os.listdir(SD)))


## CELL 8 - Install/upgrade and run the fast tests

In [ ]:
# ===== CELL 8: INSTALL + FAST TESTS =====
!pip -q install -U duckdb scipy scikit-learn matplotlib pytest
import scipy; print("scipy", scipy.__version__)      # must be >= 1.10
os.chdir(REPO)
!python -m pytest tests/test_events.py tests/test_series.py tests/test_smoothers.py tests/test_benchmark.py tests/test_model.py tests/test_usability.py -q


## CELL 9 - Cross-check our labelling against Kaggle's train.csv

In [ ]:
# ===== CELL 9: LABEL CROSS-CHECK (SEND ME THE OUTPUT) =====
os.chdir(REPO)
!python -m silentchurn.pipeline validate --sample-dir {SD} --work {WORK} --force


## CELL 10 - Build the event table and the engagement matrix

In [ ]:
# ===== CELL 10: EVENTS + SERIES (resumable; add --force to redo) =====
os.chdir(REPO)
!python -m silentchurn.pipeline events --sample-dir {SD} --work {WORK}
!python -m silentchurn.pipeline series --sample-dir {SD} --work {WORK}


## CELL 11 - Detection lead-time benchmark (RQ1; the long step)

In [ ]:
# ===== CELL 11: BENCHMARK =====
# Runs the automated causality test on every smoother first, then the bootstrap (B=200 resamples of customers).
os.chdir(REPO)
!python -m silentchurn.pipeline benchmark --sample-dir {SD} --work {WORK} --B 200


## CELL 12 - The one simple model, drift and subgroup tables

In [ ]:
# ===== CELL 12: MODEL =====
os.chdir(REPO)
!python -m silentchurn.pipeline model --sample-dir {SD} --work {WORK}


## CELL 13 - Power BI tables, static fallback pages, usability-study package

In [ ]:
# ===== CELL 13: EXPORT + FIGURES + USABILITY MATERIALS =====
os.chdir(REPO)
!python -m silentchurn.pipeline export --sample-dir {SD} --work {WORK} --force
!python -m silentchurn.pipeline figures --sample-dir {SD} --work {WORK} --force
!python -m silentchurn.pipeline usability --sample-dir {SD} --work {WORK} --participants 12 --force
from IPython.display import Image, display
for i in range(1, 6):
    display(Image(f"{WORK}/figures/page{i}.png"))


## CELL 14 - Download everything to your PC

In [ ]:
# ===== CELL 14: ZIP + DOWNLOAD =====
import shutil, os
stage = '/content/silentchurn_package'
shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage)
for name in ['powerbi', 'figures', 'usability']:
    shutil.copytree(f'{WORK}/{name}', f'{stage}/{name}')
for f in ['benchmark.csv', 'benchmark_counts.csv', 'validate.json']:
    shutil.copy(f'{WORK}/{f}', stage)
z = shutil.make_archive('/content/silentchurn_package', 'zip', stage)
print(z, round(os.path.getsize(z) / 1e6, 1), "MB")
from google.colab import files
files.download(z)
